In [1]:
import numpy as np
import os
# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import time
import cudf as cd 

In [2]:
passmark = 40
start = time.time()

In [3]:
### cell 0 ###
cudf_df = cd.read_csv("/home/dias-benchmarks/notebooks/spscientist/student-performance-in-exams/input/StudentsPerformance.csv")

In [4]:
### cell 1 ###

factor = 1000
cudf_df = cd.concat([cudf_df]*factor)
cudf_df.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 1000000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                       Non-Null Count    Dtype
---  ------                       --------------    -----
 0   gender                       1000000 non-null  object
 1   race/ethnicity               1000000 non-null  object
 2   parental level of education  1000000 non-null  object
 3   lunch                        1000000 non-null  object
 4   test preparation course      1000000 non-null  object
 5   math score                   1000000 non-null  object
 6   reading score                1000000 non-null  object
 7   writing score                1000000 non-null  object
dtypes: object(8)
memory usage: 83.8+ MB


In [ ]:
### cell 2 ###

df.head()

In [ ]:
### cell 3 ###

print(df.shape)

In [ ]:
### cell 4 ###

df.describe()

In [ ]:
### cell 5 ###

df.isnull().sum()

In [ ]:
### cell 6 ###

df['math score'] = df['math score'].astype(float).fillna(-1)  # Replace NaNs with a default value

# Vectorized conditional assignment using cuDF
df['Math_PassStatus'] = (df['math score'] >= passmark).astype("str").replace({'True': 'P', 'False': 'F'})

# Use cuDF's optimized value_counts()
df['Math_PassStatus'].value_counts()

In [ ]:
### cell 7 ###

#rewritten
# Convert 'reading score' to numeric (cuDF does not have errors='coerce', so use fillna after)
df['reading score'] = df['reading score'].astype(float).fillna(-1)  # Replace NaNs if needed

# Use cuDF's vectorized conditional assignment
df['Reading_PassStatus'] = (df['reading score'] >= passmark).astype("str").replace({'True': 'P', 'False': 'F'})

# Use cuDF's value_counts() (faster than pandas on GPU)
df['Reading_PassStatus'].value_counts()


In [ ]:
### cell 8 ###

#rewritten
# Convert 'reading score' to numeric (cuDF does not have errors='coerce', so use fillna after)
df['reading score'] = df['reading score'].astype(float).fillna(-1)  # Replace NaNs if needed

# Use cuDF's vectorized conditional assignment
df['Reading_PassStatus'] = (df['reading score'] >= passmark).astype("str").replace({'True': 'P', 'False': 'F'})

# Use cuDF's value_counts() (faster than pandas on GPU)
df['Reading_PassStatus'].value_counts()

##rewritten
df['writing score'] = df['writing score'].astype('float32')

# Use cuDF's boolean indexing + direct assignment (avoids unnecessary operations)
df['Writing_PassStatus'] = 'F'  # Default all to 'F' first
df.loc[df['writing score'] >= passmark, 'Writing_PassStatus'] = 'P'  # Assign 'P' where needed

# Efficient value_counts() with cuDF
df['Writing_PassStatus'].value_counts()

In [ ]:
### cell 9 ###

df['OverAll_PassStatus'] = ((df['Math_PassStatus'] == 'P') & 
                            (df['Reading_PassStatus'] == 'P') & 
                            (df['Writing_PassStatus'] == 'P')).astype("str").replace({'True': 'P', 'False': 'F'})

# Optimized cuDF value_counts()
df['OverAll_PassStatus'].value_counts()

In [ ]:
### cell 10 ###

df['Total_Marks'] = df['math score']+df['reading score']+df['writing score']
df['Percentage'] = df['Total_Marks']/3

In [ ]:
### cell 11 ###

##rewritten
# Initialize Grade column with 'F' (default for failures)
df['Grade'] = 'F'

# Apply vectorized conditional assignments
df.loc[df['OverAll_PassStatus'] == 'F', 'Grade'] = 'F'
df.loc[(df['Percentage'] >= 80) & (df['OverAll_PassStatus'] != 'F'), 'Grade'] = 'A'
df.loc[(df['Percentage'] >= 70) & (df['OverAll_PassStatus'] != 'F'), 'Grade'] = 'B'
df.loc[(df['Percentage'] >= 60) & (df['OverAll_PassStatus'] != 'F'), 'Grade'] = 'C'
df.loc[(df['Percentage'] >= 50) & (df['OverAll_PassStatus'] != 'F'), 'Grade'] = 'D'
df.loc[(df['Percentage'] >= 40) & (df['OverAll_PassStatus'] != 'F'), 'Grade'] = 'E'

# Efficient value counts
df['Grade'].value_counts()
